**Author:** Pankaj Yadav          
**Date:** 04/19/2026                 
**Description:**              
This assignment is to learn connecting GenAI API and using Retrieval Augmented Generation (RAG). 

**Instructions**     
Create a chat like the one in this week’s lecture video using any Wikipedia page except the one in the video. Be sure to ask your bot at least two separate questions. 

Build the solution using a Jupyter Notebook (no other Python is acceptable) and include Markdown that thoroughly explains your thought process and your commentary on the results you achieved. Obvious comments that explicitly only explain the code will be ignored since I can read the code. I’m looking for your commentary and evaluation in your own words.

I setup os, chromadb, pprint, operator, and OpenAI-related libraries for this notebook workflow. os.getenv helps me fetch the API key from environment variables instead of hardcoding it, pprint helps display outputs in a clean readable way, and index is used while creating unique IDs for stored document chunks. WikipediaLoader is used to pull source content, RecursiveCharacterTextSplitter breaks it into smaller chunks, Chroma and chromadb store and retrieve embeddings, OpenAIEmbeddings converts text into vectors, PromptTemplate formats the question context, ChatOpenAI connects to the model for responses, and RetrievalQA combines retrieval plus LLM answering in one chain.

In [113]:
import os
import chromadb
import pprint
from operator import index
from openai import OpenAI
from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_openai.chat_models import ChatOpenAI

I set the search topic to IPL 2026 which is the  cricket tournament and used WikipediaLoader to pull content directly from Wikipedia so the chatbot can answer questions from a real, up-to-date source. The query uses load_max_docs=1 to focus on one main article and avoid mixing unrelated pages, while doc_content_chars_max=300000 allows a much larger portion of the page to be captured. This larger limit is important because match-by-match details are usually listed much lower in the article, and a small character cap can miss those sections.

In [114]:
search_term = "IPL 2026"
docs = WikipediaLoader(query=search_term, load_max_docs=1, doc_content_chars_max=300000).load()

I use RecursiveCharacterTextSplitter to break the Wikipedia article into smaller, meaningful chunks before creating embeddings. I set chunk_size to 1000 so each piece is small enough for accurate retrieval, and chunk_overlap to 20 so important details that fall near boundaries are not lost between chunks. length_function=len measures chunk size by characters, and is_separator_regex=False keeps splitting behavior simple and stable using normal separators. This setup improves retrieval quality for specific match-level questions in the RAG pipeline.

In [115]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000
                                               , chunk_overlap = 20
                                               , length_function = len
                                               , is_separator_regex=False, )

In this step, I split the loaded Wikipedia content into smaller document chunks using the text splitter configuration defined earlier, and store those chunks in data. Running data right after that helps me quickly inspect the output structure, confirm chunking worked correctly, and verify that important sections from the source page are present before moving to embeddings and vector storage. This checkpoint is useful for validating retrieval quality early in the RAG pipeline.

In [116]:
data = text_splitter.split_documents(docs)
data

[Document(metadata={'title': '2026 Indian Premier League', 'summary': 'The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, is the 19th edition of the Indian Premier League, a professional Twenty20 cricket league. The tournament features 10 teams competing in 74 matches. It is being played from 28 March to 31 May across 13 venues.\nRoyal Challengers Bengaluru are the defending champions.', 'source': 'https://en.wikipedia.org/wiki/2026_Indian_Premier_League'}, page_content='The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, is the 19th edition of the Indian Premier League, a professional Twenty20 cricket league. The tournament features 10 teams competing in 74 matches. It is being played from 28 March to 31 May across 13 venues.\nRoyal Challengers Bengaluru are the defending champions.\n\n\n== Background ==\nThe Indian Premier League (IPL) is a professional Twenty20 (T20) cricket league, organized by the Board of Control for 

In this step, I securely load the OpenAI API key from environment variables using os.getenv instead of exposing it directly in the notebook. Then I initialize OpenAIEmbeddings with that key so each text chunk can be converted into vector embeddings for semantic search in the Chroma database. This is a core RAG step because retrieval quality depends on how well these embeddings represent the meaning of the source content.

In [117]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(api_key=api_key)

In this step, I create the Chroma vector store from the chunked Wikipedia data and generated embeddings so the chatbot can retrieve relevant context during question answering. I also assign custom IDs using each chunk’s source metadata plus its index to keep records unique and traceable. The collection is named IPL_2026 for clear organization, and persist_directory='db' saves the vector database locally so I can reuse it later without rebuilding embeddings every time I run the notebook.

In [118]:

store = Chroma.from_documents(
    documents=data,
    embedding=embeddings,
    ids = [f"{item.metadata['source']}-{index}" for index, item in enumerate(data)],
    collection_name="IPL_2026",
    persist_directory='db',
)

In this step, I define the prompt template that controls how the LLM should answer questions in the RAG flow. The template tells the model to rely on retrieved context, answer the user question directly, and return “I don’t know” when the information is missing, which helps reduce hallucinations. The placeholders {context} and {question} are dynamically filled at runtime with retrieved chunks and the user query, so the response stays grounded in the indexed Wikipedia content.

In [119]:
template = """You are the bot that answers questions about the Indian Premier League (IPL) 2026. 
            Use the following pieces of context to answer the question at the end. If you don't know 
            the answer, say you don't know. Always use all available data to answer the question.
            {context}
            Question: {question}"""

In this step, I convert the raw prompt text into a reusable PromptTemplate object so it can be injected cleanly into the RetrievalQA chain. The input variables context and question map to the retrieved chunks and the user query at runtime, ensuring each response is generated using both relevant evidence and the exact question asked. This makes the prompt dynamic, structured, and consistent across all chatbot queries.

In [120]:
prompt = PromptTemplate(
    template = template,
    input_variables = ["context", "question"],
    )

In this step, I initialize the chat language model that will generate final answers from the retrieved IPL context. I set temperature=0 to keep responses more deterministic and factual, which is important for RAG tasks where consistency matters. The api_key is passed securely from environment variables so the notebook can authenticate with OpenAI without hardcoding secrets.

In [121]:
llm = ChatOpenAI(model="gpt-5.4", temperature=0, api_key=api_key)   

In this step, I build the final RetrievalQA chain that connects retrieval and generation into one pipeline. The retriever pulls the most relevant chunks from the Chroma store, and the llm uses the custom prompt to answer the question using that context. I chose chain_type as stuff so the retrieved chunks are combined and sent together to the model, and return_source_documents=True helps me inspect which source chunks were used, making the output more transparent and easier to validate.

In [122]:
qa_with_context = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=store.as_retriever(),
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt,},
    return_source_documents=True,
)

In this step, I run the first user query through the complete RAG pipeline to test whether the chatbot can retrieve relevant IPL 2026 context and produce a grounded answer. The question checks a basic factual detail (“When did IPL 2026 start?”), which is a good sanity test for both retrieval quality and prompt behavior. I used pprint to display the full structured output clearly, including the answer and source documents, so I can verify not just what the model said but also where that information came from.

In [123]:
pprint.pprint(qa_with_context("When did the IPL 2026 start?"))

{'query': 'When did the IPL 2026 start?',
 'result': 'IPL 2026 started on **28 March 2026**.',
 'source_documents': [Document(metadata={'summary': 'The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, is the 19th edition of the Indian Premier League, a professional Twenty20 cricket league. The tournament features 10 teams competing in 74 matches. It is being played from 28 March to 31 May across 13 venues.\nRoyal Challengers Bengaluru are the defending champions.', 'title': '2026 Indian Premier League', 'source': 'https://en.wikipedia.org/wiki/2026_Indian_Premier_League'}, page_content='The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, is the 19th edition of the Indian Premier League, a professional Twenty20 cricket league. The tournament features 10 teams competing in 74 matches. It is being played from 28 March to 31 May across 13 venues.\nRoyal Challengers Bengaluru are the defending champions.\n\n\n== Background ==\nThe

In this step, I run a second evaluation query to test whether the RAG system can retrieve and summarize a more nuanced topic, not just a simple date or match fact.

In [124]:
pprint.pprint(qa_with_context("What were the controversies regarding playersin IPL 2026?"))

{'query': 'What were the controversies regarding playersin IPL 2026?',
 'result': 'The player-related controversy mentioned for IPL 2026 was about '
           'Bangladeshi fast bowler Mustafizur Rahman.\n'
           '\n'
           '- He was bought by Kolkata Knight Riders for ₹9.2 crore in the '
           'auction.\n'
           '- After anti-Hindu violence in Bangladesh during 2025 and the '
           'lynching of Dipu Chandra Das in December 2025, there were calls to '
           'exclude Bangladeshi players from the IPL.\n'
           '- In January 2026, the BCCI asked the franchise to release '
           'Mustafizur Rahman from the squad and allowed them to choose a '
           'replacement.\n'
           '- This decision became controversial and drew mixed reactions:\n'
           '  - Criticized by Madan Lal, Shashi Tharoor, Khaled Mahmud, and '
           'Mohammad Ashraful.\n'
           '  - Defended by Aakash Chopra and BJP politician Sangeet Singh '
           'Som.\n

In this step, I ask a targeted fact-check question to verify whether the RAG pipeline can retrieve a specific fixture-level detail from deep within the IPL 2026 page. This is a test to make sure RAG chain is working correctly and refusing to guess.

In [125]:
pprint.pprint(qa_with_context("Who won the 'Match 28' to be played on '19 April 2026' in the IPL 2026?"))

{'query': "Who won the 'Match 28' to be played on '19 April 2026' in the IPL "
          '2026?',
 'result': "I don't know.\n"
           '\n'
           'The context only provides general IPL 2026 tournament details, '
           'scheduling background, and fixture announcement information. It '
           'does not include the result of Match 28 on 19 April 2026.',
 'source_documents': [Document(metadata={'summary': 'The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, is the 19th edition of the Indian Premier League, a professional Twenty20 cricket league. The tournament features 10 teams competing in 74 matches. It is being played from 28 March to 31 May across 13 venues.\nRoyal Challengers Bengaluru are the defending champions.', 'title': '2026 Indian Premier League', 'source': 'https://en.wikipedia.org/wiki/2026_Indian_Premier_League'}, page_content='The 2026 Indian Premier League, also known as IPL 19 and branded as TATA IPL 2026, is the 19th editio

In this diagnostic step, I directly check whether the phrase Match 28 exists in the chunked documents before retrieval. The first line returns a boolean to confirm presence at least once, and the second line returns the exact number of chunks containing that phrase. This helps isolate whether the issue is in ingestion and chunking or in retrieval ranking: as the result is False and 0, the relevant section was never indexed.

In [126]:
print(any("Match 28" in d.page_content for d in data))
print(sum("Match 28" in d.page_content for d in data))

False
0
